## Block 1 · Imports

In [ ]:
# --- Standard library: file paths and reproducibility ---
import os
import random
from pathlib import Path

# --- Data handling: arrays and tables ---
import numpy as np
import pandas as pd

# --- Plots: MFCC images, training curves, confusion matrix ---
import matplotlib.pyplot as plt

# --- Audio: load WAV, resample to 16 kHz, compute MFCC/MFE, save clips ---
import librosa
import soundfile as sf

# --- Neural network: layers, training, TFLite int8 export ---
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split

# --- Evaluation: per-class metrics and confusion matrix ---
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# --- Reproducibility: same results on every run ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# --- Paths ---
DATA_DIR = Path("data")
RESULTS_DIR = Path("results")
EXPORT_DIR = Path("exported")

# --- Audio and features (same values as in Edge Impulse) ---
SAMPLE_RATE = 16000      # Hz, standard for speech
CLIP_SECONDS = 1.0       # one command fits in 1 s
FRAME_LENGTH = 0.02      # 20 ms frames
FRAME_STRIDE = 0.01      # 10 ms step
N_MFCC = 13              # coefficients per frame
N_MELS = 32              # mel filters

# --- Classes: order defines the network's output indices ---
CLASSES = ["telegram", "elly", "slack", "focus", "noise", "unknown"]

# --- Training ---
EPOCHS = 100
LEARNING_RATE = 0.005
BATCH_SIZE = 32

# --- Check that everything is installed ---
print("TensorFlow:", tf.__version__)
print("librosa:", librosa.__version__)

## Block 2 · Dataset

In [ ]:
# --- Parse file name: label.speaker.number.wav -> label, speaker ---
def parse_name(path):
    parts = path.stem.split(".")
    if len(parts) < 3:
        return None
    return {"path": path, "label": parts[0].lower(), "speaker": parts[1].lower()}

# --- Build the dataset index: one row per clip ---
all_wavs = sorted(DATA_DIR.rglob("*.wav"))
rows = [r for p in all_wavs if (r := parse_name(p))]
skipped = [p.name for p in all_wavs if parse_name(p) is None]
df = pd.DataFrame(rows, columns=["path", "label", "speaker"])

# --- Keep only the classes we train on ---
ignored = sorted(set(df["label"]) - set(CLASSES))
df = df[df["label"].isin(CLASSES)].reset_index(drop=True)

# --- Summary: how many clips per class and per speaker ---
print(f"Clips: {len(df)}")
if skipped:
    print("Skipped, wrong file name:", skipped[:10])
if ignored:
    print("Ignored labels:", ignored)
if df.empty:
    print(f"No clips found in {DATA_DIR.resolve()}")
else:
    display(pd.crosstab(df["label"], df["speaker"], margins=True))
    assert df["label"].nunique() == len(CLASSES), "not all classes present"
    assert df["label"].value_counts().min() >= 2, "stratify needs >= 2 clips per class"
    assert df["speaker"].nunique() >= 2, "need >= 2 speakers for the held-out test"

## Block 3 · Sound

No code here. Sound is a pressure wave; the microphone measures it 16,000 times per second, so one second becomes 16,000 numbers. See the interactive page for block 3 on the site.

## Block 4 · Clips

In [ ]:
# --- Target length: 1 s at 16 kHz = 16000 samples ---
N_SAMPLES = int(SAMPLE_RATE * CLIP_SECONDS)

# --- Load one clip: resample to 16 kHz, mono, exactly 1 s ---
def load_clip(path):
    y, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    if len(y) > N_SAMPLES:
        # longer than 1 s: keep the loudest 1 s window (where the word is)
        cs = np.concatenate([[0], np.cumsum(y ** 2)])
        energy = cs[N_SAMPLES:] - cs[:-N_SAMPLES]
        start = int(np.argmax(energy))
        y = y[start:start + N_SAMPLES]
    else:
        # shorter than 1 s: pad with silence at the end
        y = np.pad(y, (0, N_SAMPLES - len(y)))
    return y.astype(np.float32)

# --- Load all clips into one array: (number of clips, 16000) ---
if len(df):
    X_audio = np.stack([load_clip(p) for p in df["path"]])
else:
    X_audio = np.empty((0, N_SAMPLES), dtype=np.float32)
print("Audio array:", X_audio.shape)

# --- Quick check: waveform of the first clip ---
if len(df):
    plt.figure(figsize=(10, 2.5))
    plt.plot(np.arange(N_SAMPLES) / SAMPLE_RATE, X_audio[0])
    plt.title(f'{df.loc[0, "label"]} ({df.loc[0, "speaker"]})')
    plt.xlabel("time, s")
    plt.tight_layout()
    plt.show()

## Block 5 · MFCC

In [ ]:
WIN = int(SAMPLE_RATE * FRAME_LENGTH)   # 320 samples
HOP = int(SAMPLE_RATE * FRAME_STRIDE)   # 160 samples
N_FFT = 512

def compute_mfcc(y):
    m = librosa.feature.mfcc(y=y, sr=SAMPLE_RATE, n_mfcc=N_MFCC, n_mels=N_MELS,
                             n_fft=N_FFT, win_length=WIN, hop_length=HOP,
                             window="hann", center=False)
    return m.T.astype(np.float32)       # (frames, 13)

In [ ]:
class_to_idx = {c: i for i, c in enumerate(CLASSES)}
assert len(X_audio), "no clips loaded: check the data/ folder"
X = np.stack([compute_mfcc(a) for a in X_audio])   # (N, 99, 13)
y = df["label"].map(class_to_idx).values.astype(np.int64)
print(X.shape, y.shape)

## Block 6 · Split

In [ ]:
HELD_OUT_SPEAKER = df["speaker"].unique()[0]     # change by hand to choose who is the test speaker
test_mask = (df["speaker"] == HELD_OUT_SPEAKER).values
assert test_mask.any() and (~test_mask).any(), "empty train or test"
X_trainval, y_trainval = X[~test_mask], y[~test_mask]
X_test, y_test = X[test_mask], y[test_mask]
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.15, stratify=y_trainval, random_state=SEED)
print(len(X_train), len(X_val), len(X_test))
print("test per class:", np.bincount(y_test, minlength=len(CLASSES)))

In [ ]:
mean = X_train.mean(axis=(0, 1), keepdims=True)
std = X_train.std(axis=(0, 1), keepdims=True) + 1e-6
X_train, X_val, X_test = [(a - mean) / std for a in (X_train, X_val, X_test)]
X_train, X_val, X_test = [a[..., np.newaxis] for a in (X_train, X_val, X_test)]

## Block 7 · Model

In [ ]:
def build_model(input_shape, n_classes):
    return keras.Sequential([
        keras.layers.Input(shape=input_shape),
        keras.layers.Conv2D(8, (3, 3), padding="same", activation="relu"),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Conv2D(16, (3, 3), padding="same", activation="relu"),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Dropout(0.25),
        keras.layers.GlobalAveragePooling2D(),
        keras.layers.Dense(n_classes, activation="softmax"),
    ])

model = build_model(X_train.shape[1:], len(CLASSES))
model.summary()

## Block 8 · Training

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
early = keras.callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)
history = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                    epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=[early], verbose=2)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(history.history["loss"], label="train"); ax[0].plot(history.history["val_loss"], label="val"); ax[0].set_title("loss"); ax[0].legend()
ax[1].plot(history.history["accuracy"], label="train"); ax[1].plot(history.history["val_accuracy"], label="val"); ax[1].set_title("accuracy"); ax[1].legend()
plt.tight_layout(); plt.show()

## Block 9 · Evaluation

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"test accuracy (held-out speaker): {test_acc:.3f}")
y_pred = model.predict(X_test, verbose=0).argmax(axis=1)
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred, labels=range(len(CLASSES))), display_labels=CLASSES).plot(xticks_rotation=45)
plt.show()
print(classification_report(y_test, y_pred, labels=range(len(CLASSES)), target_names=CLASSES, zero_division=0))

## Block 10 · Quantization

In [ ]:
def representative_data():
    for i in range(min(200, len(X_train))):
        yield [X_train[i:i+1].astype(np.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
tflite_model = converter.convert()
EXPORT_DIR.mkdir(exist_ok=True)
(EXPORT_DIR / "exs_int8.tflite").write_bytes(tflite_model)
np.savez(EXPORT_DIR / "norm.npz", mean=mean, std=std)
print(f"model size: {len(tflite_model)/1024:.1f} KB")

In [ ]:
interp = tf.lite.Interpreter(model_content=tflite_model); interp.allocate_tensors()
inp, out = interp.get_input_details()[0], interp.get_output_details()[0]
s, z = inp["quantization"]
correct = 0
for x, yt in zip(X_test, y_test):
    q = np.clip(np.round(x[np.newaxis] / s + z), -128, 127).astype(np.int8)
    interp.set_tensor(inp["index"], q); interp.invoke()
    correct += int(interp.get_tensor(out["index"]).argmax() == yt)
print(f"int8 test accuracy: {correct/len(y_test):.3f}")

## Block 11 · Prediction and deployment

In [ ]:
def predict_file(path):
    x = (compute_mfcc(load_clip(path))[np.newaxis] - mean) / std   # (1, 99, 13)
    x = x[..., np.newaxis]                                         # (1, 99, 13, 1)
    p = model.predict(x, verbose=0)[0]
    return CLASSES[int(p.argmax())], float(p.max())